In [6]:
import numpy as np
from numba import cuda
from tensorflow.keras.datasets import mnist

hist_bins = 256

@cuda.jit
def histogram_shared(img,hist):

    shared_hist = cuda.shared.array(256,dtype=np.int32)

    tid = cuda.threadIdx.x

    if tid < 256:
        shared_hist[tid] = 0

    cuda.syncthreads()

    idx = cuda.grid(1)

    if idx < img.size:
        pixel = img[idx]
        cuda.atomic.add(shared_hist,pixel,1)

    cuda.syncthreads()

    if tid < 256:
        cuda.atomic.add(hist,tid,shared_hist[tid])


(trainX,_),(_,_) = mnist.load_data()

image = trainX[0].flatten().astype(np.uint8)

hist = np.zeros(256,dtype=np.int32)

d_img = cuda.to_device(image)
d_hist = cuda.to_device(hist)

threads = 256
blocks = (image.size + threads - 1)//threads

histogram_shared[blocks,threads](d_img,d_hist)

hist = d_hist.copy_to_host()

print(hist)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[618   2   3   1   0   0   0   0   0   1   0   3   0   0   1   0   2   0
   5   0   0   0   0   1   1   1   1   1   0   0   1   0   0   0   0   1
   1   0   0   2   0   0   0   1   0   1   1   0   0   1   0   0   0   0
   0   1   1   0   0   0   0   0   0   0   2   0   1   0   0   0   1   0
   0   0   0   0   0   0   1   0   2   2   2   0   0   0   0   0   0   0
   1   0   0   2   1   0   0   0   0   0   0   0   0   0   0   0   0   1
   1   0   0   0   0   0   1   0   0   0   0   1   0   0   0   0   0   0
   1   1   0   0   1   0   1   1   0   1   2   0   0   1   0   0   0   0
   0   0   0   0   1   0   1   0   0   0   3   0   1   0   0   0   1   0
   0   0   0   0   1   0   0   0   1   1   2   0   0   1   0   0   0   0
   0   0   2   1   0   0   1   1   0   0   2   0   0   0   0   2   0   0
   2   0   0   1   0   0   0   1   0   1   0   0   0   0   1   1   0   0
   0   0   0   2   0   1   0   0   0   2   1   0   0   1   0   0   0   0


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
